## 지식 그래프 구축

In [10]:
from neo4j import GraphDatabase 
import pandas as pd 
import time 
import os 

NEO4J_URI = "bolt://localhost:7687"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = "password"
NEO4J_DATABASE = "neo4j"
GRAPHRAG_FOLDER = "./working_directory/output"

In [3]:
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD), 
    connection_timeout=5,
)

In [11]:
print(driver.verify_connectivity())
print(os.listdir(GRAPHRAG_FOLDER))

None
['text_units.parquet', 'documents.parquet', 'relationships.parquet', 'communities.parquet', 'context.json', 'community_reports.parquet', 'entities.parquet', 'stats.json', 'lancedb']


In [7]:
records, summary, keys = driver.execute_query("RETURN 1 AS result")
print(records)
print(summary)
print(keys)

[<Record result=1>]
['result']


In [8]:
def batched_import(statement, df, batch_size=1000):
    """
    Import a dataframe into Neo4j using a batched approach.

    Parameters: statement is the Cypher query to execute, df is the dataframe to import, and batch_size is the number of rows to import in each batch.
    """
    total = len(df)
    start_s = time.time()
    for start in range(0, total, batch_size):
        batch = df.iloc[start : min(start + batch_size, total)]
        result = driver.execute_query(
            "UNWIND $rows AS value " + statement,
            rows=batch.to_dict("records"),
            database_=NEO4J_DATABASE,
        )
        print(result.summary.counters)
    print(f"{total} rows in {time.time() - start_s} s.")
    return total

In [9]:
statements = [
    "\ncreate constraint chunk_id if not exists for (c:__Chunk__) require c.id is unique",
    "\ncreate constraint document_id if not exists for (d:__Document__) require d.id is unique",
    "\ncreate constraint community_id if not exists for (c:__Community__) require c.community is unique",
    "\ncreate constraint entity_id if not exists for (e:__Entity__) require e.id is unique",
    "\ncreate constraint entity_title if not exists for (e:__Entity__) require e.name is unique",
    "\ncreate constraint covariate_title if not exists for (e:__Covariate__) require e.title is unique",
    "\ncreate constraint related_id if not exists for ()-[rel:RELATED]->() require rel.id is unique",
    "\n",
]

for statement in statements:
    if len((statement or "").strip()) > 0:
        print(statement)
        driver.execute_query(statement)


create constraint chunk_id if not exists for (c:__Chunk__) require c.id is unique

create constraint document_id if not exists for (d:__Document__) require d.id is unique

create constraint community_id if not exists for (c:__Community__) require c.community is unique

create constraint entity_id if not exists for (e:__Entity__) require e.id is unique

create constraint entity_title if not exists for (e:__Entity__) require e.name is unique

create constraint covariate_title if not exists for (e:__Covariate__) require e.title is unique

create constraint related_id if not exists for ()-[rel:RELATED]->() require rel.id is unique


In [34]:
doc_df = pd.read_parquet(
    f"{GRAPHRAG_FOLDER}/documents.parquet", columns=["id", "title"]
)
# print(doc_df.columns.tolist())

statement = """
MERGE (d:__document__ {id:value.id})
SET d += value {.title}
"""

batched_import(statement, doc_df)

SummaryCounters{properties_set: 1, contains_updates: True, contains_system_updates: False}
1 rows in 0.11093306541442871 s.


1

In [35]:
# 텍스트 유닛 임포트 
text_df = pd.read_parquet(f"{GRAPHRAG_FOLDER}/text_units.parquet", 
                          columns=["id", "text", "n_tokens", "document_id"])

statement = """
MERGE (c: __Chunk__ {id:value.id})
SET c += value {.text, .n_tokens}
WITH c, value 
UNWIND value.document_ids as document
MATCH (d: __Document__ {id:document})
MERGE (c)-[:PART_OF]->(d)
"""

batched_import(statement, text_df)


SummaryCounters{labels_added: 29, nodes_created: 29, properties_set: 87, contains_updates: True, contains_system_updates: False}
29 rows in 0.4373619556427002 s.


29

In [36]:
# 엔티티 임포트 
#entity_df = pd.read_parquet(f"{GRAPHRAG_FOLDER}/entities.parquet")
#print(entity_df.columns.to_list())

entity_df = pd.read_parquet(f"{GRAPHRAG_FOLDER}/entities.parquet",
                            columns=["title", "type", "description", "human_readable_id", "id", "text_unit_ids"])

statement = """
MERGE (e:__Entity__ {id: value.id})
SET e.human_readable_id = value.human_readable_id,
    e.description = value.description,
    e.name = coalesce(replace(value.title, '"', ''), 'Unknown')
WITH e, value
CALL apoc.create.addLabels(e, CASE WHEN coalesce(value.type, "") = "" THEN [] ELSE [apoc.text.upperCamelCase(replace(value.type, '"', ''))] END) YIELD node
UNWIND value.text_unit_ids AS text_unit
MATCH (c:__Chunk__ {id: text_unit})
MERGE (c)-[:HAS_ENTITY]->(e)
"""
batched_import(statement, entity_df)


SummaryCounters{labels_added: 84, relationships_created: 97, nodes_created: 84, properties_set: 336, contains_updates: True, contains_system_updates: False}
84 rows in 0.3270070552825928 s.


84

In [39]:
# 관계 임포트 
rel_df = pd.read_parquet(f"{GRAPHRAG_FOLDER}/relationships.parquet")
# print(rel_df.columns.to_list())

rel_df = pd.read_parquet(f"{GRAPHRAG_FOLDER}/relationships.parquet",
                         columns=["source", "target", "id", "combined_degree", "weight", "human_readable_id", "description", "text_unit_ids"])

rel_df.rename(columns={"combined_degree":"rank"})

rel_statement = """
    MATCH (source:__Entity__ {name:replace(value.source,'"','')})
    MATCH (target:__Entity__ {name:replace(value.target,'"','')})
    MERGE (source)-[rel:RELATED {id: value.id}]->(target)
    SET rel += value {.rank, .weight, .human_readable_id, .description, .text_unit_ids}
    RETURN count(*) as createdRels
"""

batched_import(rel_statement, rel_df)


SummaryCounters{relationships_created: 59, properties_set: 295, contains_updates: True, contains_system_updates: False}
59 rows in 0.1690528392791748 s.


59

In [40]:
# 커뮤니티 임포트
# cmmunity_df = pd.read_parquet(f'{GRAPHRAG_FOLDER}/communities.parquet')
# print(community_df.columns.to_list())

community_df = pd.read_parquet(
    f'{GRAPHRAG_FOLDER}/communities.parquet',
    columns=["id", "level", "title", "text_unit_ids", "relationship_ids"]
)

statement = """
MERGE (c:__Community__ {community: value.title})
SET c.title = value.title,
    c.level = value.level
WITH c, value
UNWIND value.text_unit_ids as text_unit_id
MATCH (t:__Chunk__ {id: text_unit_id})
MERGE (c)-[:HAS_CHUNK]->(t)
WITH distinct c, value
UNWIND value.relationship_ids as rel_id
MATCH (start:__Entity__)-[:RELATED {id: rel_id}]->(end:__Entity__)
MERGE (start)-[:IN_COMMUNITY]->(c)
MERGE (end)-[:IN_COMMUNITY]->(c)
RETURN count(distinct c) as createdCommunities
"""

batched_import(statement, community_df)

SummaryCounters{labels_added: 3, relationships_created: 19, nodes_created: 3, properties_set: 9, contains_updates: True, contains_system_updates: False}
3 rows in 0.15068411827087402 s.


3

In [41]:
# 커뮤니티 보고서 임포트
community_report_df = pd.read_parquet(
    f'{GRAPHRAG_FOLDER}/community_reports.parquet',
    columns=["id", "community", "level", "title", "summary", "findings", "rank", "rating_explanation", "full_content"]
)

community_report_df['community'] = "Community " + community_report_df['community'].astype(str)

community_statement = """
MERGE (c:__Community__ {community: value.community})
SET c.level = value.level,
    c.name = value.title,
    c.rank = value.rank,
    c.rank_explanation = value.rating_explanation,
    c.full_content = value.full_content,
    c.summary = value.summary
WITH c, value
UNWIND range(0, size(value.findings)-1) AS finding_idx
WITH c, value, finding_idx, value.findings[finding_idx] AS finding
MERGE (c)-[:HAS_FINDING]->(f:Finding {id: finding_idx})
SET f += finding
"""

batched_import(community_statement, community_report_df)



SummaryCounters{labels_added: 9, relationships_created: 9, nodes_created: 9, properties_set: 45, contains_updates: True, contains_system_updates: False}
3 rows in 0.10137414932250977 s.


3